# Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_ollama import ChatOllama
os.environ["OLLAMA_API_KEY"] = os.getenv("OLLAMA_API_KEY")

## Creating a LangChain Agent with the GEHC GenAI Gateway

This section explains how to connect a LangChain agent to a model hosted on the
internal GEHC GenAI gateway using the OpenAI-compatible client.

### Imports

| Import | Purpose |
|--------|---------|
| `os` | Read environment variables (gateway URL and API key). |
| `load_dotenv` | Load values from the `.env` file into the environment. |
| `ChatOpenAI` | OpenAI-compatible chat client (required for the LiteLLM `:4000` gateway). |
| `create_agent` | Wraps a model into a runnable LangGraph agent. |

### `load_dotenv()`

Reads the `.env` file so `os.getenv(...)` can access `GENAI_BASE_URL` and
`OLLAMA_API_KEY`. Without this call, both values return `None`.

### Model configuration — `ChatOpenAI`

| Parameter | Description |
|-----------|-------------|
| `model="llama3.2"` | Model to use. Must be on the approved gateway model list. |
| `base_url` | GEHC gateway endpoint (`http://genai-imaging-lab...:4000/v1`). |
| `api_key` | Gateway API key used for authentication. |
| `temperature=0` | Controls randomness. `0` = deterministic; higher = more creative. |

### Agent creation — `create_agent`

| Parameter | Description |
|-----------|-------------|
| `model=llm` | The configured `ChatOpenAI` instance. |
| `tools=[]` | No tools attached — a plain chat agent. |
| `system_prompt` | Persona/instructions prepended to every conversation. |

### Important Notes

- `print("Agent Created Successfully!!")` confirms the objects were built in
  memory only. **No network call occurs at this stage.**
- The model, endpoint, and API key are validated only when `agent.invoke(...)`
  is called.
- Ensure `OLLAMA_API_KEY` in `.env` holds the **GEHC gateway key**, not an
  external `ollama.com` key.

### Execution Flow

```text
.env file
   │  (load_dotenv reads it)
   ▼
ChatOpenAI  ──►  connects to llama3.2 on the GEHC gateway
   │
   ▼
create_agent  ──►  wraps model + system prompt into an agent
   │
   ▼
"Agent Created Successfully!!"   (built, not yet called)
   │
   ▼
agent.invoke({...})   ← the actual request happens here
```



In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

load_dotenv()

llm = ChatOpenAI(
    model="llama3.2",
    base_url=os.getenv("GENAI_BASE_URL"),  
    api_key=os.getenv("OLLAMA_API_KEY"),
    temperature=0,
)

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant",
)
print("Agent Created Successfully!!")

Agent Created Successfully!!


### Invoke the agent 

```python
agent.invoke({...})
```

- This runs your agent once, synchronously, and waits for the full response. invoke is the "give me the final answer" method (as opposed to stream, which yields tokens as they arrive).

#### What happens internally
```mermaid
flowchart TD
    A[User message] --> B[System prompt prepended:<br/>'You are a helpful assistant']
    B --> C[Sent to gemma3:27b<br/>via GEHC GenAI gateway]
    C --> D[Model generates reply<br/>'assistant' message]
    D --> E[Reply appended to<br/>messages list]
```

In [5]:
result = agent.invoke({
    "messages": [                 
        {"role": "user", "content": "What is Langchain"}
    ]
})
print(result["messages"][-1].content)

Langchain is an open-source, Python-based library that enables the creation of custom, modular, and extensible knowledge graphs. It's designed to facilitate the development of large-scale, complex knowledge graphs by providing a flexible and scalable framework for integrating multiple data sources.

Langchain was created with the goal of making it easier to build and manage knowledge graphs, which are essential in various applications such as:

1. **Question Answering Systems**: Langchain can be used to create custom question answering systems that integrate multiple knowledge graph sources.
2. **Natural Language Processing (NLP)**: It provides a way to represent and manipulate knowledge graphs in a more structured and accessible format for NLP tasks.
3. **Information Retrieval**: Langchain can be used to build custom search engines or information retrieval systems that leverage knowledge graphs.

Some key features of Langchain include:

1. **Modular Architecture**: Langchain is design